# Optional workflow: disaggregate aSoCC MRIO sectors


Disaggregate non LCIA deterministic allocated shares of carrying capacities (aSoCC). The function
starts from a target source available at an aggregated sector resolution and publishes a new
disaggregated source. It uses a reference ixi MRIO available at both the same aggregated
sector resolution and the requested detailed sector resolution to distribute the target
aggregated sector across detailed sectors.


## Workspace setup

Run `set_workspace(...)` before any later function call. It sets the active workspace root
for the current Python session, creates or reuses the workspace, imports package
prerequisites under `data_raw/`, and records setup guidance in `data_raw/summary.log`.

In [ ]:
from pyaesa import set_workspace

# Windows example; update this path before running.
set_workspace(r"C:\Users\username\Documents\aesa_workspace")
# macOS example; update this path before running.
# set_workspace("/Users/username/Documents/aesa_workspace")

Methodological notes from the repository methodological notes folder are imported with package
prerequisites:

* `data_raw/methodological_notes/` contains notes for definition
  of functional units and allocation methods, prospective allocation, and
  uncertainty sources.
* `data_raw/carrying_capacities/` contains the note for definition of steady
  state and dynamic carrying capacities.


The function writes a published disaggregated aSoCC source from a target source
available at an aggregated sector resolution. It uses a reference ixi MRIO available
at both the same aggregated sector resolution and the requested detailed sector
resolution to distribute each target aggregated sector across the detailed sectors.
Figures are rendered by default (`figures=True`). Use `figures=False` to skip them.

## Public argument checklist
The table lists all arguments; the same definitions are available in the function docstring.

<div class="pyaesa-argument-legend">
<div class="pyaesa-default-block" style="color:#087f5b"><strong>Green items = default if omitted.</strong></div>
<div class="pyaesa-optional-block" style="color:#c45f00"><strong>Orange items = optional feature skipped if omitted.</strong></div>
</div>

Do not write green or orange items when that behavior is intended.

<details open>
<summary><code>disaggregate_asocc(...)</code> arguments</summary>

<table>
<thead><tr><th>Argument</th><th>Description</th></tr></thead>
<tbody>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>disaggregation_config</code></td><td>Required disaggregation envelope. Required keys<br>&#10;are <code>target_agg_run</code>, <code>ref_agg_run</code>,<br>&#10;<code>ref_disagg_run</code>, <code>disaggregation_specs</code>, and<br>&#10;<code>new_disagg_version_name</code>.<br>&#10;&#160;<br>&#10;Disaggregation configuration fields:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>target_agg_run</code>: aggregated deterministic aSoCC source to<br>&#10;  disaggregate. Its published rows supply <code>target_aggregated</code> in<br>&#10;  the formula above. Example: OECD ICIO sector <code>D</code>.<br>&#10;&bull;&nbsp;<code>ref_agg_run</code>: reference ixi source aggregated to the same<br>&#10;  sector labels as <code>target_agg_run</code>. Its published rows<br>&#10;  supply <code>ref_aggregated</code>. Example: EXIOBASE ixi aggregated to OECD<br>&#10;  ICIO sector <code>D</code>.<br>&#10;&bull;&nbsp;<code>ref_disagg_run</code>: the same reference source as<br>&#10;  <code>ref_agg_run</code>, but at the detailed disaggregated sector labels<br>&#10;  that should be written in the new source. Its published rows<br>&#10;  supply <code>ref_disaggregated</code>. Example: EXIOBASE ixi electricity sectors.<br>&#10;&bull;&nbsp;<code>disaggregation_specs</code>: mapping from each aggregated sector label<br>&#10;  to the detailed disaggregated sector label(s) that replace it in the new<br>&#10;  source.<br>&#10;&bull;&nbsp;<code>new_disagg_version_name</code>: output source label used for<br>&#10;  the published disaggregated aSoCC source created by this<br>&#10;  function.<br>&#10;&#160;<br>&#10;Each <code>*_run</code> selector requires:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>source</code>: supported MRIO source key for this transformation.<br>&#10;  Accepted values are <code>&quot;oecd_v2025&quot;</code>, <code>&quot;exiobase_3102_ixi&quot;</code>,<br>&#10;  and <code>&quot;exiobase_396_ixi&quot;</code>. Only ixi MRIO layouts are supported.<br>&#10;&bull;&nbsp;<code>s_p</code>: non empty list of sector labels.<br>&#10;<span style="color:#c45f00">&bull;&nbsp;<code>agg_reg</code>: If <code>True</code>, reclassify MRIO regions with the <code>agg_reg_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping. The mapping can keep native labels, aggregate several native regions into one target label, or disaggregate one native region across several target labels when a <code>weight</code> column is provided.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source regions.</span><br>&#10;<span style="color:#c45f00">&bull;&nbsp;<code>agg_sec</code>: If <code>True</code>, reclassify MRIO sectors with the <code>agg_sec_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping. The mapping can keep native labels, aggregate several native sectors into one target label, or disaggregate one native sector across several target labels when a <code>weight</code> column is provided.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source sectors.</span><br>&#10;<span style="color:#c45f00">&bull;&nbsp;<code>agg_version</code>: Name token used to resolve the matching <code>agg_reg_&lt;agg_version&gt;.csv</code> and/or <code>agg_sec_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping files in <code>data_raw/mrio/&lt;source&gt;/aggregation</code>. Required when <code>agg_reg</code> or <code>agg_sec</code> is True.<br>&#10;<strong>Defaults to</strong> an empty string for native source classification. Use the same token in downstream calls that should reuse the processed classification. If a mapping file has a <code>weight</code> column, weights must sum to <code>1</code> for each original label.</span><br>&#10;&#160;<br>&#10;<code>disaggregation_specs</code> is a non empty list of<br>&#10;<code>{&quot;agg_sector_label&quot;: ..., &quot;disagg_sector_label&quot;: ...}</code><br>&#10;mappings. <code>target_agg_run.s_p</code> and<br>&#10;<code>ref_agg_run.s_p</code> must exactly match the aggregated labels,<br>&#10;<code>ref_disagg_run.s_p</code> must exactly match the disaggregated labels,<br>&#10;<code>ref_agg_run.source</code> must equal <code>ref_disagg_run.source</code>,<br>&#10;and one disaggregated sector can map to exactly one aggregated sector.<br>&#10;The studied region labels requested through <code>r_p</code>, <code>r_c</code>, or<br>&#10;<code>r_f</code> must be present with the same names in all three selected<br>&#10;prerequisite scopes. If source native region names differ, use<br>&#10;<code>agg_reg=True</code> and an aggregation version to rename or aggregate<br>&#10;regions before disaggregation.<br>&#10;</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>base_asocc_args</code></td><td>Deterministic aSoCC scope used to match prerequisite<br>&#10;published outputs and define the written disaggregated output scope.<br>&#10;Write nested arguments as <code>base_asocc_args={&quot;project_name&quot;: &quot;...&quot;, &quot;fu_code&quot;: &quot;L2.c.b&quot;}</code>. Source, MRIO aggregation and disaggregation, and sector identity are<br>&#10;owned by <code>disaggregation_config</code>. LCIA selectors are not<br>&#10;accepted, and the scope must resolve non LCIA deterministic aSoCC<br>&#10;outputs.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>project_name</code>: Required project name used to build<br>&#10;  <code>&lt;repo&gt;/&lt;project_name&gt;</code>.<br>&#10;&bull;&nbsp;<code>fu_code</code>: Required functional unit code (for example<br>&#10;  <code>&quot;L1.a&quot;</code>, <code>&quot;L2.c.b&quot;</code>). See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;  for all available functional unit codes and the system<br>&#10;  boundaries each represents. Disaggregation is defined only on<br>&#10;  L2 published outputs.<br>&#10;&bull;&nbsp;<code>years</code>: Studied years. Accepts a single year, list, or<br>&#10;  range. <strong>If omitted</strong>, all available MRIO years for the selected<br>&#10;  source and <code>agg_version</code> are used.<br>&#10;&bull;&nbsp;<code>r_p</code>: Producing region filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid producing regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code><br>&#10;  file.<br>&#10;&bull;&nbsp;<code>r_c</code>: Consuming region filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid consuming regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code><br>&#10;  file.<br>&#10;&bull;&nbsp;<code>r_f</code>: Final demand region filter(s), single string or list.<br>&#10;  If this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid final demand regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code><br>&#10;  file.<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>group_indices</code>: Whether multiple selected region or sector filter values are kept as separate result rows or summed into one result row after the function calculation has been performed.<br>&#10;&bull;&nbsp;<code>False</code> (<strong>default</strong>): keep selected values as independent rows.<br>&#10;&bull;&nbsp;<code>True</code>: sum selected values into one result row.<br>&#10;The function refuses to run when <code>group_indices=True</code> is used with <code>L2.a.b</code>, <code>L2.b.b</code>, or <code>L2.c.b</code> because summing output rows for CBA total demand boundaries can double count. For these functional units, change the upstream MRIO aggregation and disaggregation scope with <code>agg_reg</code>, <code>agg_sec</code>, and <code>agg_version</code> before running the study.</span><br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>method_plan</code>: <code>method_plan</code> <strong>defaults to</strong> <code>&quot;default&quot;</code> and<br>&#10;  accepts <code>&quot;default&quot;</code>, <code>&quot;one_step&quot;</code>, <code>&quot;two_steps&quot;</code>,<br>&#10;  <code>&quot;pairs&quot;</code>, or <code>&quot;one_step_pairs&quot;</code>. <strong>When omitted</strong>, all <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span><br>&#10;  allocation methods available for the selected <code>fu_code</code> are<br>&#10;  applied. See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;  for the allocation methods available per functional unit,<br>&#10;  including definitions and mathematical expressions.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l1_methods</code>: Optional L1 subset. <strong>Omit</strong> it to keep all L1<br>&#10;  methods allowed by <code>method_plan</code>. In <code>&quot;default&quot;</code>, this<br>&#10;  filters only L1 weights used by two step methods. In<br>&#10;  <code>&quot;two_steps&quot;</code>, it filters the two step cartesian L1 side.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>one_step_methods</code>: Optional one step L2 subset. <strong>Omit</strong> it to<br>&#10;  keep all one step methods allowed by <code>method_plan</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>two_step_methods</code>: Optional two step L2 subset. <strong>Omit</strong> it to<br>&#10;  keep all two step L2 methods allowed by <code>method_plan</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l1_l2_pairs</code>: Explicit pair list formatted as<br>&#10;  <code>&quot;L1METHOD::L2METHOD&quot;</code>. <strong>Omit</strong> it unless <code>method_plan</code> is<br>&#10;  <code>&quot;pairs&quot;</code> or <code>&quot;one_step_pairs&quot;</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l1_reg_aggreg</code>: L1 aggregation mode for methods where timing<br>&#10;  matters (<code>PR(GDPcap)</code>, <code>PR-HR(Ecap)</code> and <code>AR(Ecap)</code>).<br>&#10;  <code>&quot;pre&quot;</code> aggregates regions before L1 computation. <code>&quot;post&quot;</code><br>&#10;  (<strong>default</strong>) computes on original regions, then aggregates.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>ssp_scenario</code>: SSP scenario name or list. <strong>Defaults to</strong><br>&#10;  <code>[&quot;SSP1&quot;, &quot;SSP2&quot;, &quot;SSP3&quot;, &quot;SSP4&quot;, &quot;SSP5&quot;]</code> and is applied<br>&#10;  when scenario dependent inputs are required.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>projection_mode</code>: Projection policy for post historical<br>&#10;  years of L2 utilitarian (UT) methods (MRIO economic enacting<br>&#10;  metrics). <strong>Defaults to</strong> <code>&quot;regression&quot;</code>. <code>&quot;regression&quot;</code><br>&#10;  projects UT inputs for future years. <code>&quot;historical_reuse&quot;</code><br>&#10;  reuses historical UT structures.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>reg_window</code>: Historical regression fit window for regression<br>&#10;  mode. Provide it as <code>range(start_year, end_year + 1)</code> or as<br>&#10;  an explicit list of consecutive years in fit window order. When<br>&#10;  <strong>omitted</strong>, the source registry supplies the <strong>default</strong> fit window<br>&#10;  from the modeled year minimum through the source historical<br>&#10;  cutoff. For EXIOBASE 3.10.2 and OECD ICIO v2025, this resolves<br>&#10;  to 1995 to 2022; other supported MRIO sources use their own<br>&#10;  registry window.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>l2_reuse_years</code>: Historical L2 reuse year selector used by<br>&#10;  all UT historical reuse routes. In<br>&#10;  <code>projection_mode=&quot;historical_reuse&quot;</code> it applies to all UT<br>&#10;  methods; in <code>projection_mode=&quot;regression&quot;</code> it applies to<br>&#10;  adjusted UT routes (<code>UT(FDa)</code>, <code>UT(GVAa)</code>), which always<br>&#10;  use historical reuse as regression is not supported (would<br>&#10;  require regression on full MRIO). <strong>If omitted</strong>, <strong>defaults to</strong><br>&#10;  <code>reg_window</code> when required.<br>&#10;</span></td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>output_format</code></td><td>Persisted output file format: <code>&quot;csv&quot;</code> (<strong>default</strong>),<br>&#10;<code>&quot;pickle&quot;</code>, or <code>&quot;parquet&quot;</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figures</code></td><td>Whether to render figures.<br>&#10;<strong>Default is</strong> <code>True</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_format</code></td><td>Figure render settings mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;format&quot;: &quot;png&quot;, &quot;dpi&quot;: 500}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>format</code>: Figure file format. Accepted values are <code>&quot;png&quot;</code>,<br>&#10;  <code>&quot;pdf&quot;</code>, and <code>&quot;svg&quot;</code>.<br>&#10;&bull;&nbsp;<code>dpi</code>: Positive integer figure resolution used for raster<br>&#10;  outputs.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_external_method</code></td><td>Optional external deterministic aSoCC selector<br>&#10;block used only for figure rendering. Use<br>&#10;<code>prepare_external_inputs(...)</code> to import the external aSoCC runnable CSV examples and <code>README_external_asocc_templates.txt</code>, then follow that README for<br>&#10;method syntax and data input format. This argument is valid only<br>&#10;when <code>figures=True</code>. <strong>Omit</strong> it to render only native<br>&#10;deterministic aSoCC method rows. <strong>Defaults to</strong> <code>None</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>refresh</code></td><td>If <code>True</code>, remove and rebuild only the published disaggregated aSoCC source created by this call. The cleared scope is the <code>deterministic</code> folder under <code>&lt;project&gt;/B1_asocc/&lt;new_disagg_version_name&gt;</code>. The deterministic prerequisite scopes named in <code>target_agg_run</code>, <code>ref_agg_run</code>, and <code>ref_disagg_run</code> are not refreshed. Processed MRIO inputs, processed population and GDP, raw downloads, and downstream aCC or ASR outputs are not refreshed. <strong>Defaults to</strong> <code>False</code>.</td></tr>
</tbody>
</table>

</details>


Only non LCIA aSoCC methods are supported. Do not pass `lcia_method` in
`base_asocc_args`. Supported sources are `oecd_v2025`, `exiobase_3102_ixi`,
and `exiobase_396_ixi`. Region labels are matched strictly between MRIO
sources. Use region aggregation to update region names syntax when they do not
already match.


## Prerequisite process and deterministic aSoCC scopes

The disaggregation call reads existing published aSoCC outputs. Run the three
processed MRIO scopes first: target aggregated, reference aggregated, and reference
disaggregated. The target and reference aggregated scopes must use the same sector
resolution. The reference disaggregated scope must use the detailed sector resolution
that will be written in the new published source. Raw MRIO archives must already exist
from `download_mrio(...)`.


In [ ]:
from pyaesa import process_mrio

# Target aggregated source: OECD ICIO sector D.
# Region aggregation example fr renames the OECD ISO3 label "FRA"
# to the EXIOBASE ISO2 label "FR" so region labels match across sources.
target_aggregated_mrio_report = process_mrio(
    source="oecd_v2025",
    years=2022,
    agg_reg=True,
    agg_version="fr",
)
print(target_aggregated_mrio_report)
print()

# Reference aggregated source: EXIOBASE ixi aggregated to OECD ICIO sector D.
ref_aggregated_mrio_report = process_mrio(
    source="exiobase_3102_ixi",
    years=2022,
    agg_sec=True,
    agg_version="oecd_d",
)
print(ref_aggregated_mrio_report)
print()

# Reference disaggregated source: EXIOBASE ixi at the detailed electricity resolution.
ref_disaggregated_mrio_report = process_mrio(
    source="exiobase_3102_ixi",
    years=2022,
    agg_sec=True,
    agg_version="elec",
)
print(ref_disaggregated_mrio_report)

Then publish the three matching deterministic aSoCC scopes. These calls must
use the same year, functional unit, region selector, and sector labels that
will be referenced by `disaggregation_config`.

Use [tutorials/study_objectives/1_functional_units_and_allocation_methods.md](../study_objectives/1_functional_units_and_allocation_methods.md) to choose the compatible FU and selector arguments for the deterministic aSoCC scopes used by this workflow.


In [ ]:
from pyaesa import deterministic_asocc

# target_agg_run
# Region aggregation example fr renames the OECD ISO3 label "FRA"
# to the EXIOBASE ISO2 label "FR" so region labels match across sources.
target_aggregated_asocc_report = deterministic_asocc(
    project_name="elec_fr_demo",
    source="oecd_v2025",
    agg_reg=True,
    agg_version="fr",
    years=range(1995, 2031),
    fu_code="L2.c.b",
    s_p=["D"],
    r_c=["FR"],
    figures=False,
)
print(target_aggregated_asocc_report)
print()

# ref_agg_run
ref_aggregated_asocc_report = deterministic_asocc(
    project_name="elec_fr_demo",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="oecd_d",
    years=range(1995, 2031),
    fu_code="L2.c.b",
    s_p=["D"],
    r_c=["FR"],
    figures=False,
)
print(ref_aggregated_asocc_report)
print()

# ref_disagg_run
ref_disaggregated_asocc_report = deterministic_asocc(
    project_name="elec_fr_demo",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=range(1995, 2031),
    fu_code="L2.c.b",
    s_p=["Electricity"],
    r_c=["FR"],
    figures=False,
)
print(ref_disaggregated_asocc_report)

## Disaggregate OECD ICIO sector D into EXIOBASE electricity sector


In [ ]:
from pyaesa import disaggregate_asocc

disaggregate_asocc(
    disaggregation_config={
        "target_agg_run": {
            "source": "oecd_v2025",
            # Region aggregation example fr renames the OECD ISO3 label "FRA"
            # to the EXIOBASE ISO2 label "FR" so region labels match across sources.
            "agg_reg": True,
            "agg_version": "fr",
            "s_p": ["D"],
        },
        "ref_agg_run": {
            "source": "exiobase_3102_ixi",
            "agg_sec": True,
            "agg_version": "oecd_d",
            "s_p": ["D"],
        },
        "ref_disagg_run": {
            "source": "exiobase_3102_ixi",
            "agg_sec": True,
            "agg_version": "elec",
            "s_p": ["Electricity"],
        },
        "disaggregation_specs": [
            {
                "agg_sector_label": "D",
                "disagg_sector_label": "Electricity",
            }
        ],
        "new_disagg_version_name": "oecd_electricity",
    },
    base_asocc_args={
        "project_name": "elec_fr_demo",
        "years": range(1995, 2031),
        "fu_code": "L2.c.b",
        "r_c": ["FR"],
    },
)